# grad-tracking-global-toggle — worked example 2: no_grad decorator with try/finally restore

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-tracking-global-toggle`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

A decorator-style `no_grad` wraps a function so it runs with gradient tracking disabled. It must snapshot and restore the module-level flag using `try/finally` — the `finally` block guarantees restoration even if the decorated function raises an exception. Without `finally`, a crash inside the decorated function would leave the flag disabled for all subsequent code in the process.

## Worked solution

**Step 1 — snapshot the current flag.** Before calling the function, read `grad_tracking_enabled` and save it in a local variable `prev`.

**Step 2 — disable tracking.** Set `grad_tracking_enabled = False`. Using the `global` keyword here is mandatory — without it, Python creates a local variable in the wrapper's scope and the module-level flag never changes.

**Step 3 — call function with try/finally.** The `try` block calls `fn(*args, **kwargs)` and returns its result. The `finally` block restores `grad_tracking_enabled = prev` regardless of what happens.

**Step 4 — test exception safety.** A function that raises still has the flag restored. We verify this by catching the exception and checking the global state afterwards.

In [ ]:
import torch as t

grad_tracking_enabled = True

def _get_flag():
    return globals()['grad_tracking_enabled']

def no_grad(fn):
    def wrapper(*args, **kwargs):
        global grad_tracking_enabled
        prev = grad_tracking_enabled
        grad_tracking_enabled = False
        try:
            return fn(*args, **kwargs)
        finally:
            grad_tracking_enabled = prev
    return wrapper

@no_grad
def inference_step(x):
    return x * 2

@no_grad
def crashing_step(x):
    raise ValueError("simulated error")

# Normal decorated call: flag is False inside, restored to True after
print(f"Before: {_get_flag()}")  # True
result = inference_step(t.randn(3))
print(f"After normal call: {_get_flag()}")  # True

# Exception inside decorated fn: flag still restored
try:
    crashing_step(t.randn(3))
except ValueError:
    pass
print(f"After exception in decorated fn: {_get_flag()}")  # True
assert _get_flag() == True, "Flag must be restored even after exception!"
print("All checks passed.")